# 03 · Lo que no se puede contar

**Duración estimada:** 25 minutos

## Propósito

Los tres cuadernos anteriores construyeron una representación que funciona: tokenizar,
armar el vocabulario, contar, pesar. Este cuaderno mide **dónde deja de funcionar**.

No por un error de configuración ni por una lista de stopwords mal armada. Por lo que la
representación **es**: bag of words no compara significados, compara **cadenas de
caracteres**. Todo lo que el idioma hace con las palabras —decir lo mismo de dos maneras,
decir dos cosas con la misma palabra— le pasa por al lado.

Vamos a ponerle número a cada una de esas fallas. Después, en la unidad de spaCy, vas a
ver cuáles se arreglan y cuál no.

## Qué consume

Nada. Las mismas cuatro oraciones del `00`, y unas pocas más que vamos a escribir acá.

## ◈ Microglosario

| Término | Qué es en lenguaje llano |
|---|---|
| **Forma** | La palabra como cadena de caracteres, sin su significado |
| **Sinonimia** | Dos formas distintas para el mismo significado: *padre* y *papá* |
| **Polisemia** | Una forma con varios significados: *banco* |
| **Campo semántico** | Un grupo de palabras que hablan de lo mismo: *cancha*, *hinchada*, *gol* |
| **Similitud coseno** | Un número de 0 a 1 que dice cuánto se parecen dos vectores |
| **Tesauro** | Una lista de términos donde está escrito **cómo se relacionan entre sí** |

## 1. Antes de medir lo que falta, hay que poder medir «parecido»

**La analogía.** Cada documento es una flecha que sale del origen. Lo que nos interesa no
es su largo —un texto largo tiene una flecha larga— sino **hacia dónde apunta**. Dos
documentos que hablan de lo mismo apuntan para el mismo lado.

La **similitud coseno** mide justamente eso: el ángulo entre dos flechas, convertido en un
número de 0 a 1.

| Valor | Qué significa |
|---|---|
| 1 | Las dos flechas apuntan exactamente igual |
| 0 | No tienen **ninguna** palabra en común |

**Dónde vive esto.** Es la medida con la que un buscador ordena resultados, con la que un
sistema recomienda "artículos relacionados" y con la que se detectan textos duplicados.
Cada vez que un programa dice *"esto se parece a esto otro"*, casi siempre es un coseno.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

# cosine_similarity compara todos los documentos contra todos, de una.
from sklearn.metrics.pairwise import cosine_similarity

import pandas as pd

# Las mismas cuatro oraciones del cuaderno 00.
documentos = [
    "El perro muerde al hombre",
    "El hombre muerde al perro",
    "El perro come carne",
    "El hombre come comida",
]

nombres_de_fila = ["Doc 1", "Doc 2", "Doc 3", "Doc 4"]

Vectorizamos como en el `01` y le pedimos el coseno.

In [ ]:
contador = CountVectorizer()
matriz = contador.fit_transform(documentos)

parecidos = cosine_similarity(matriz)

tabla_de_parecidos = pd.DataFrame(parecidos, columns=nombres_de_fila, index=nombres_de_fila)
tabla_de_parecidos.round(3)

Leé la celda de arriba a la derecha: **Doc 1 contra Doc 2 da 1,000.**

*El perro muerde al hombre* y *El hombre muerde al perro* son, para esta medida,
**el mismo documento**. Ya lo sabíamos desde el `01`; ahora tiene número, y el número es el
máximo posible.

Los otros valores son razonables: Doc 3 y Doc 4 dan 0,500 porque comparten `el` y `come`,
y los pares cruzados dan 0,447.

> **Ojo con lo que mide el coseno.** No mide significado. Mide **cuánto vocabulario
> comparten dos documentos**. Todo lo que sigue en este cuaderno son consecuencias de esa
> frase.

## 2. Sinonimia: el falso negativo

**La analogía.** Dos personas cuentan el mismo accidente. Una dice *"un auto chocó"*, la
otra dice *"un coche se estrelló"*. Un testigo entiende que hablan de lo mismo. Un buscador
de palabras exactas, no.

Escribamos la misma oración del Doc 1 con otras palabras, sin repetir ni una sola de
contenido.

In [ ]:
mismo_significado = [
    "El perro muerde al hombre",
    "El can ataca al señor",
]

contador_sinonimos = CountVectorizer()
matriz_sinonimos = contador_sinonimos.fit_transform(mismo_significado)

parecido = cosine_similarity(matriz_sinonimos)[0, 1]

print("vocabulario:", list(contador_sinonimos.get_feature_names_out()))
print()
print("similitud:", round(parecido, 3))

**0,4.** Y lo interesante es de dónde sale ese 0,4: las dos oraciones comparten exactamente
dos palabras, `el` y `al`.

O sea que **lo único que estas dos oraciones tienen en común, para la máquina, es
justamente lo que en el `01` decidimos tirar a la basura.** Saquémoslo, entonces.

In [ ]:
# La lista del cuaderno 01, más las que hacen falta para las oraciones nuevas.
palabras_vacias = ["el", "al", "la", "los", "las", "un", "una", "de", "y",
                   "en", "me", "a", "que", "con", "es", "fue", "mi", "por", "todo"]

contador_sin_vacias = CountVectorizer(stop_words=palabras_vacias)
matriz_sin_vacias = contador_sin_vacias.fit_transform(mismo_significado)

parecido_sin_vacias = cosine_similarity(matriz_sin_vacias)[0, 1]

print("vocabulario:", list(contador_sin_vacias.get_feature_names_out()))
print()
print("similitud:", round(parecido_sin_vacias, 3))

**Cero.** El mínimo posible, el mismo valor que le daría a dos textos sobre temas que no
tienen nada que ver.

Y no es un caso de laboratorio: pasa con cualquier par de sinónimos. Midámoslo palabra por
palabra, tratando cada palabra como si fuera un documento de una sola palabra.

In [ ]:
pares_de_sinonimos = [
    ["padre", "papá"],
    ["auto", "coche"],
    ["hermoso", "lindo"],
    ["enojado", "podrido"],
]

filas = []
for par in pares_de_sinonimos:
    contador_del_par = CountVectorizer()
    matriz_del_par = contador_del_par.fit_transform(par)
    valor = cosine_similarity(matriz_del_par)[0, 1]
    filas.append({"palabra": par[0], "sinónimo": par[1], "similitud": round(valor, 3)})

pd.DataFrame(filas)

Cero en los cuatro.

**Para bag of words, `padre` y `papá` están exactamente tan lejos como `padre` y
`heladera`.** No es que se equivoque por poco: le da el mismo número al par de sinónimos y
al par sin relación alguna.

Esto se llama **falso negativo**: dos cosas que son parecidas, medidas como distintas.

## 3. Polisemia: el falso positivo

El error también funciona en el otro sentido, y es más peligroso porque no se ve.

`banco` es una forma con al menos tres significados. Escribamos una oración con cada uno:
una habla de plata, otra de una plaza y la tercera de fútbol.

### ✎ Antes de ejecutar

Las tres oraciones no tienen **nada** que ver entre sí. ¿Qué similitud esperás que dé la
celda siguiente? Anotá tu número antes de correrla.

In [ ]:
tres_bancos = [
    "Fui al banco a sacar plata",
    "Me senté en el banco de la plaza",
    "El banco de suplentes estaba lleno",
]

contador_bancos = CountVectorizer(stop_words=palabras_vacias)
matriz_bancos = contador_bancos.fit_transform(tres_bancos)

nombres = ["dinero", "plaza", "fútbol"]
parecidos_bancos = cosine_similarity(matriz_bancos)

pd.DataFrame(parecidos_bancos, columns=nombres, index=nombres).round(3)

Las tres oraciones se parecen entre sí: entre 0,250 y 0,289.

No es mucho, pero **no es cero**, y en un ranking de mil documentos ese valor alcanza para
subir un texto y bajar otro. Lo importante es de dónde sale.

In [ ]:
vocabulario_bancos = contador_bancos.get_feature_names_out()

tabla_bancos = pd.DataFrame(
    matriz_bancos.toarray(),
    columns=vocabulario_bancos,
    index=nombres,
)
tabla_bancos

**La única columna que tiene un 1 en las tres filas es `banco`.**

Una oración habla de plata, otra de una plaza y la tercera de fútbol. No tienen nada que
ver entre sí, y el sistema las relaciona por una palabra que en cada caso significa una
cosa distinta.

Esto es un **falso positivo**, y es el error más difícil de detectar de los dos: el número
no llama la atención, y quien lee el resultado no tiene manera de saber que salió de una
coincidencia de forma.

> **Lo vas a encontrar en el corpus real.** En el anexo de esta unidad, la palabra `che`
> aparece veinte veces en doce años de textos de Casciari. Seis de esas veinte no son la
> interjección rioplatense: son Ernesto *Che* Guevara. El conteo sobreestima el marcador
> oral en un treinta por ciento, y ninguna cuenta de frecuencias puede avisarlo.

### Los dos errores, uno al lado del otro

| | Qué pasa con la forma | Qué pasa con el significado | Cómo lo mide bag of words |
|---|---|---|---|
| **Sinonimia** | Dos formas distintas | El mismo | Distintos: **falso negativo** |
| **Polisemia** | La misma forma | Dos significados | Parecidos: **falso positivo** |

Los dos errores son la misma cosa dicha de dos maneras: **la forma y el significado no se
corresponden, y bag of words solo puede ver la forma.**

## 4. Campo semántico: cuando todo da cero

Sinonimia y polisemia son casos extremos. Lo habitual es más suave: dos textos que hablan
de lo mismo **sin usar ninguna palabra igual**, porque el idioma tiene muchas maneras de
rodear un tema.

Tres oraciones de fútbol que no comparten ni una palabra de contenido, y una cuarta que no
tiene nada que ver.

In [ ]:
cuatro_textos = [
    "El partido de fútbol fue emocionante",
    "La hinchada cantó todo el encuentro",
    "La tribuna se volvió loca con el gol",
    "Cada relato es una invitación al lector",
]

etiquetas = ["fútbol 1", "fútbol 2", "fútbol 3", "escritura"]

contador_campos = CountVectorizer(stop_words=palabras_vacias)
matriz_campos = contador_campos.fit_transform(cuatro_textos)

parecidos_campos = cosine_similarity(matriz_campos)
pd.DataFrame(parecidos_campos, columns=etiquetas, index=etiquetas).round(3)

Mirá la tabla completa: **fuera de la diagonal, todo es cero.**

Las tres oraciones de fútbol dan 0,000 entre sí. La oración de escritura da 0,000 contra
las tres. **Los dos casos reciben el mismo número.**

Y ahí está el problema, que es peor que un error de precisión:

> **No es que el número esté mal. Es que es el mismo número.** Con estos vectores no se
> puede *ordenar*: no hay forma de decir "estos tres textos están más cerca entre sí que
> del cuarto", porque la medida no distingue "relacionado" de "no relacionado" cuando no
> hay palabras compartidas.

Cualquier buscador armado así deja afuera todo lo que se dijo con otras palabras. Y en un
corpus de verdad —doce años de textos de una persona que escribe bien y evita repetirse—
eso es muchísimo.

### ✎ Para pensar

- En el práctico vas a comparar años del corpus de Casciari por su vocabulario. Con lo que
  acabás de medir: ¿qué tipo de cambio entre dos años **no** vas a poder detectar?
- Si un buscador de jurisprudencia funcionara así, ¿qué fallo tendría al buscar
  *"despido"* en un texto que dice *"cese de la relación laboral"*?

## 5. Lo que sí se puede hacer sin salir de acá

Los tres problemas tienen arreglos parciales que no requieren ninguna herramienta nueva.
Ninguno alcanza, y por razones distintas.

| Arreglo | Qué resuelve | Qué cuesta |
|---|---|---|
| **N-gramas** (visto en el `02`) | Algo del orden: `perro muerde` contra `hombre muerde` | Multiplica el vocabulario, y no toca la sinonimia |
| **Recortar terminaciones** | Une `jugar`, `jugaba`, `jugó` en una sola forma | Mutila palabras, y `padre` y `papá` siguen separados |
| **Un tesauro** | La sinonimia, de verdad | Hay que escribirlo a mano, y no termina nunca |

El tercero merece que lo probemos, porque enseña algo sobre los otros dos.

### Qué es un tesauro

**La analogía.** Pensá en el catálogo de una biblioteca antes de las computadoras. Si el
bibliotecario ficha un libro bajo *papá* y vos buscás *padre*, el libro no existe para vos.
La solución no fue técnica: fue **acordar de antemano una lista de términos válidos** y
anotar, al lado de cada uno, cuáles son sus variantes. Esa lista es un tesauro.

No es un diccionario: **un tesauro no define palabras, dice cómo se vinculan.** En su forma
completa —la que norma la ISO 25964— cada término trae tres tipos de relación:

| Relación | Sigla clásica | Qué dice | Ejemplo |
|---|---|---|---|
| Equivalencia | `USE` | Cuál es el término que se usa | `papá` USE `padre` |
| Jerarquía | `TG` / `TE` | Cuál es más general y cuál más específico | `perro` TG `mamífero`; TE `ovejero alemán` |
| Asociación | `TR` | Qué otro término está cerca sin ser sinónimo | `cancha` TR `hinchada` |

Fijate que esas tres relaciones son **exactamente los tres problemas de este cuaderno**: la
equivalencia es la sinonimia de la sección 2, y la jerarquía y la asociación son el campo
semántico de la sección 4.

**Dónde vive esto.** Los tesauros no son historia: se siguen usando y se siguen escribiendo.

| Tesauro | Para qué |
|---|---|
| **WordNet** (Princeton, 1985) | El más usado en procesamiento de lenguaje. Agrupa palabras en *synsets* —conjuntos de sinónimos— y los enlaza jerárquicamente. Hay versión en español. |
| **EuroVoc** | Legislación de la Unión Europea, en todos sus idiomas. |
| **Tesauro de la UNESCO** | Educación, cultura y ciencias sociales. |
| **Tesauro ISOC** (CSIC) | En español, para ciencias sociales y humanidades. |

El nuestro va a tener una sola de las tres relaciones, la equivalencia, y ocho entradas.
Alcanza para ver la idea funcionando.

In [ ]:
# Un tesauro mínimo: cada palabra apunta a la forma que elegimos como principal.
TESAURO = {
    "papá": "padre", "viejo": "padre",
    "can": "perro", "ataca": "muerde", "señor": "hombre",
    "coche": "auto", "lindo": "hermoso", "podrido": "enojado",
}


def normalizar(texto):
    """Reemplaza cada palabra por su forma principal, si está en el tesauro."""
    unificadas = []
    for palabra in texto.lower().split():
        unificadas.append(TESAURO.get(palabra, palabra))
    return " ".join(unificadas)


for oracion in mismo_significado:
    print(normalizar(oracion))

Las dos oraciones se volvieron **la misma cadena de caracteres**. Midamos otra vez.

In [ ]:
normalizadas = []
for oracion in mismo_significado:
    normalizadas.append(normalizar(oracion))

contador_normalizado = CountVectorizer(stop_words=palabras_vacias)
matriz_normalizada = contador_normalizado.fit_transform(normalizadas)

parecido_normalizado = cosine_similarity(matriz_normalizada)[0, 1]

print("antes del tesauro :", round(parecido_sin_vacias, 3))
print("después           :", round(parecido_normalizado, 3))

De 0,000 a 1,000 con ocho líneas de diccionario.

**Funciona.** Y por eso vale la pena mirarlo de cerca, porque tiene los mismos tres
problemas que la lista de stopwords del `01`:

1. **La escribí yo, a ojo.** `can` como sinónimo de `perro` es discutible; `podrido` como
   sinónimo de `enojado` es rioplatense y no vale en otro español.
2. **No escala.** Ocho entradas cubren dos oraciones. El español tiene cientos de miles de
   formas, y los sinónimos dependen del tema, de la región y de la época.
3. **No puede tocar la polisemia por su cuenta.** Un tesauro **unifica formas**, y la
   polisemia es una forma con dos significados: habría que hacer lo contrario, *separarla*
   en dos. Los tesauros de verdad lo resuelven con calificadores —`banco (finanzas)` y
   `banco (mobiliario)` son dos términos distintos— pero eso solo funciona si **una persona
   lee el texto y elige cuál corresponde**. Un diccionario de palabra a palabra, aplicado
   automáticamente como el nuestro, no ve contexto y no puede elegir.

> **La conclusión del cuaderno.** Lo que falta no es un algoritmo mejor de conteo. Es
> **conocimiento del idioma**: saber que *papá* y *padre* nombran lo mismo, que *hinchada*
> y *tribuna* pertenecen al mismo mundo, que *banco* significa tres cosas.
>
> Ese conocimiento se puede escribir a mano —y se hizo, durante décadas— o se puede
> aprender de millones de textos. La unidad que viene usa la segunda opción: un archivo
> entrenado que ya trae ese conocimiento adentro.

### ✎ Para pensar

- El tesauro tiene `podrido` como sinónimo de `enojado`. Si el mismo tesauro se usara sobre
  un corpus de textos de España, ¿qué pasaría?
- Un modelo entrenado en millones de textos también trae decisiones de alguien: qué textos
  se usaron, de qué años, de qué país. ¿Es más confiable que mi tesauro de ocho líneas, o
  solamente **más difícil de auditar**?

## 6. Qué resuelve la unidad que viene, y qué no

En la unidad de spaCy el texto deja de ser una fila de caracteres y pasa a tener capas.
Contra la lista de fallas de este cuaderno:

| Problema | Qué lo resuelve | Dónde |
|---|---|---|
| `jugar`, `jugaba`, `jugó` cuentan como tres palabras | Lematización | Unidad 4, cuaderno `02` |
| No sabemos quién es el sujeto de `muerde` | Análisis sintáctico | Unidad 4, cuaderno `03` |
| `padre` y `papá` dan similitud cero | Vectores densos | Unidad 4, cuaderno `07` |
| `hinchada` y `tribuna` dan similitud cero | Vectores densos | Unidad 4, cuaderno `07` |
| `banco` significa tres cosas | **No se resuelve ahí** | Modelos contextuales, más adelante |

La última fila es la importante, y conviene tenerla clara antes de empezar la unidad 4 para
no esperar de más:

> **Un vector por palabra es un vector por forma.** El modelo de spaCy le asigna a `banco`
> **un solo** vector, que es una especie de promedio de todos sus significados. Sirve para
> la sinonimia y para los campos semánticos, y **no** para la polisemia.
>
> Para que `banco` tenga un vector distinto en *"fui al banco"* y en *"me senté en el
> banco"* hace falta otra clase de modelo, que calcula el vector **según la oración**. Son
> los modelos contextuales, y son la base de todo lo que se usa hoy.

Es decir: la unidad que viene tacha cuatro de las cinco filas. La quinta queda abierta a
propósito, y sabés por qué.

## Lo que hiciste hoy

- Aprendiste a medir "parecido" con la **similitud coseno**, y le pusiste número al
  problema del `01`: dos oraciones opuestas dan **1,000**.
- Mediste la **sinonimia**: dos oraciones que dicen lo mismo con otras palabras dan
  **0,000**, el mismo valor que dos textos sin relación. Es un **falso negativo**.
- Mediste la **polisemia**: tres oraciones sobre plata, una plaza y el fútbol se parecen
  entre sí, y la única razón es la palabra `banco`. Es un **falso positivo**, y no se ve.
- Mediste el **campo semántico**: tres textos de fútbol sin palabras en común dan cero, lo
  mismo que un texto de otro tema. No es que el número esté mal: es que no permite ordenar.
- Arreglaste la sinonimia con un **tesauro de ocho líneas**, y viste por qué eso no escala
  y por qué no puede tocar la polisemia.
- Sabés qué resuelve la unidad de spaCy, y cuál de los cinco problemas **no**.

## Lo que viene

El práctico: el mismo procedimiento sobre un corpus real de doce años de textos de Hernán
Casciari, donde todas estas decisiones empiezan a tener consecuencias visibles.

Y al final de la unidad, en el anexo, la operación que permite detectar a mano lo que
ninguna cuenta puede avisar: ir a leer los casos.

## Glosario de este cuaderno

| término | qué es |
|---|---|
| **forma** | La palabra como cadena de caracteres, sin su significado. |
| **sinonimia** | Dos formas distintas con el mismo significado. Da falsos negativos. |
| **polisemia** | Una forma con varios significados. Da falsos positivos. |
| **campo semántico** | Grupo de palabras que hablan de lo mismo sin ser sinónimas. |
| **similitud coseno** | Número de 0 a 1: cuánto vocabulario comparten dos documentos. |
| **falso negativo** | Decir que dos cosas parecidas son distintas. |
| **falso positivo** | Decir que dos cosas distintas son parecidas. |
| **tesauro** | Lista de términos con sus relaciones escritas: equivalencia (`USE`), jerarquía (`TG`/`TE`) y asociación (`TR`). Se escribe a mano. WordNet es el más conocido. |
| **vector denso** | Representación donde cada palabra es una lista corta de números aprendidos. |